In [1]:
import os
import torch
import torch.nn as nn
import pandas as pd
from PIL import Image
from torchvision import transforms
from torchvision.models import mobilenet_v3_small


image_path = "/Users/romeshjain/Desktop/DL Parking Allocation/Dataset/CNR-EXT_FULL_IMAGE_1000x750/FULL_IMAGE_1000x750/SUNNY/2015-11-12/camera6/2015-11-12_1046.jpg"

def parse_image_metadata(image_path):
    parts = image_path.split(os.sep)
    weather = parts[-5]
    capture_date = parts[-3]
    camera_id = parts[-2].replace("camera", "")
    return weather, capture_date, int(camera_id)


def load_and_scale_bboxes(camera_id):
    csv_path = f"/Users/romeshjain/Desktop/DL Parking Allocation/Dataset/CNR-EXT_FULL_IMAGE_1000x750/camera{camera_id}.csv"
    bbox_df = pd.read_csv(csv_path)

    
    scale_x = 1000 / 2592
    scale_y = 750 / 1944
    bbox_df['X'] = (bbox_df['X'] * scale_x).astype(int)
    bbox_df['Y'] = (bbox_df['Y'] * scale_y).astype(int)
    bbox_df['W'] = (bbox_df['W'] * scale_x).astype(int)
    bbox_df['H'] = (bbox_df['H'] * scale_y).astype(int)
    

    bbox_df['xmax'] = bbox_df['X'] + bbox_df['W']
    bbox_df['ymax'] = bbox_df['Y'] + bbox_df['H']
    
    return bbox_df


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = mobilenet_v3_small(pretrained=False)
model.classifier[3] = nn.Linear(model.classifier[3].in_features, 2)
model.load_state_dict(torch.load('mobilenet_best_model1.pth'))
model = model.to(device)
model.eval()


transform = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])


def classify_parking_slot(model, slot_image):
    slot_image = transform(slot_image).unsqueeze(0).to(device)
    with torch.no_grad():
        output = model(slot_image)
        _, predicted = torch.max(output, 1)
    return predicted.item()  # 0 for free, 1 for busy


def recommend_parking_slot(image_path):
    
    weather, capture_date, camera_id = parse_image_metadata(image_path)
    print(f"Image Metadata -> Weather: {weather}, Capture Date: {capture_date}, Camera ID: {camera_id}")
    
    bbox_df = load_and_scale_bboxes(camera_id)
    full_image = Image.open(image_path).convert("RGB")
    
    # Iterate over bounding boxes to find an available slot
    for idx, row in bbox_df.iterrows():
        xmin, ymin, xmax, ymax = row['X'], row['Y'], row['xmax'], row['ymax']
        
        # Crop the parking slot region from full image
        parking_slot = full_image.crop((xmin, ymin, xmax, ymax))
        
        # Classify the slot
        if classify_parking_slot(model, parking_slot) == 0:  # 0 means "free"
            print(f"Recommend parking slot at location: {xmin, ymin, xmax, ymax}")
            return  # Recommend the first free slot and exit
    print("No free parking slots available.")


recommend_parking_slot(image_path)


/Users/romeshjain/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:34: NotOpenSSLWarning: urllib3 v2.0 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
/Users/romeshjain/Library/Python/3.9/lib/python/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/Users/romeshjain/Library/Python/3.9/lib/python/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)


Image Metadata -> Weather: FULL_IMAGE_1000x750, Capture Date: 2015-11-12, Camera ID: 6
Recommend parking slot at location: (169, 98, 234, 163)


In [1]:
import os
import torch
import torch.nn as nn
import pandas as pd
from PIL import Image, ImageDraw
from torchvision import transforms
from torchvision.models import mobilenet_v3_small


image_path = "/Users/romeshjain/Desktop/DL Parking Allocation/Dataset/CNR-EXT_FULL_IMAGE_1000x750/FULL_IMAGE_1000x750/SUNNY/2015-12-10/camera2/2015-12-10_0915.jpg"


def parse_image_metadata(image_path):
    parts = image_path.split(os.sep)
    weather = parts[-5]
    capture_date = parts[-3]
    camera_id = parts[-2].replace("camera", "")
    return weather, capture_date, int(camera_id)


def load_and_scale_bboxes(camera_id):
    csv_path = f"/Users/romeshjain/Desktop/DL Parking Allocation/Dataset/CNR-EXT_FULL_IMAGE_1000x750/camera{camera_id}.csv"
    bbox_df = pd.read_csv(csv_path)

  
    scale_x = 1000 / 2592
    scale_y = 750 / 1944
    bbox_df['X'] = (bbox_df['X'] * scale_x).astype(int)
    bbox_df['Y'] = (bbox_df['Y'] * scale_y).astype(int)
    bbox_df['W'] = (bbox_df['W'] * scale_x).astype(int)
    bbox_df['H'] = (bbox_df['H'] * scale_y).astype(int)
    
    
    bbox_df['xmax'] = bbox_df['X'] + bbox_df['W']
    bbox_df['ymax'] = bbox_df['Y'] + bbox_df['H']
    
    return bbox_df


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = mobilenet_v3_small(pretrained=False)
model.classifier[3] = nn.Linear(model.classifier[3].in_features, 2)
model.load_state_dict(torch.load('mobilenet_best_model1.pth'))
model = model.to(device)
model.eval()


transform = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])


def classify_parking_slot(model, slot_image):
    slot_image = transform(slot_image).unsqueeze(0).to(device)
    with torch.no_grad():
        output = model(slot_image)
        _, predicted = torch.max(output, 1)
    return predicted.item() 


def recommend_parking_slot(image_path):
    
    weather, capture_date, camera_id = parse_image_metadata(image_path)
    print(f"Image Metadata -> Weather: {weather}, Capture Date: {capture_date}, Camera ID: {camera_id}")
    
    bbox_df = load_and_scale_bboxes(camera_id)
    full_image = Image.open(image_path).convert("RGB")
    draw = ImageDraw.Draw(full_image)  
    
    
    for idx, row in bbox_df.iterrows():
        xmin, ymin, xmax, ymax = row['X'], row['Y'], row['xmax'], row['ymax']
        
        
        parking_slot = full_image.crop((xmin, ymin, xmax, ymax))
        
        
        if classify_parking_slot(model, parking_slot) == 0:  
            print(f"Recommend parking slot at location: {xmin, ymin, xmax, ymax}")
            
            
            draw.rectangle([xmin, ymin, xmax, ymax], outline="green", width=3) 

            
            draw.text((xmin, ymin), "Free", fill="green")
            
            
            full_image.save("output_with_parking_slot.jpg")  
            full_image.show()  
            
            return  
    
    print("No free parking slots available.")


recommend_parking_slot(image_path)


/Users/romeshjain/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:34: NotOpenSSLWarning: urllib3 v2.0 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
/Users/romeshjain/Library/Python/3.9/lib/python/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/Users/romeshjain/Library/Python/3.9/lib/python/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)


Image Metadata -> Weather: FULL_IMAGE_1000x750, Capture Date: 2015-12-10, Camera ID: 2
Recommend parking slot at location: (64, 493, 233, 662)
